<a href="https://colab.research.google.com/github/mfletcher011/PYTHON-SCRIPTS-FOR-INVOICES/blob/main/Swatch_Group_Invoices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install necessary libraries
print("Installing libraries...")
!pip install PyMuPDF pdfplumber pandas openpyxl
print("Installation complete.")

In [ ]:
# Cell 2: Import libraries and upload your PDFs (Regex Version)
import pandas as pd
import fitz  # PyMuPDF
import pdfplumber
import re
from google.colab import files
import io

# --- Upload Files ---
print("Please upload all your PDF invoice files.")
uploaded_files = files.upload()

if uploaded_files:
    print(f"\nSuccessfully uploaded {len(uploaded_files)} file(s):")
    for filename in uploaded_files.keys():
        print(f"- {filename}")
else:
    print("\nNo files uploaded. Please run this cell again to upload your files.")

In [ ]:
# Cell 3: Loop through and process all uploaded files (Regex Version)

# This Regex is the key:
# It captures: Style, Description, Quantity (followed by "EA"), and Cost
ITEM_REGEX = re.compile(r'^([A-Z0-9]+)\s+(.*?)\s+(\d+)\s+EA\s+([\d,.]+)$')

all_invoice_data = []  # A list to hold the data from each invoice

if uploaded_files:
    print(f"Starting to process {len(uploaded_files)} file(s)...")

    for filename, file_content in uploaded_files.items():
        print(f"\n--- Processing: {filename} ---")

        try:
            # --- 1. Extract Invoice Number (This works) ---
            invoice_number = "NOT_FOUND"
            doc_fitz = fitz.open(stream=file_content, filetype="pdf")
            page1_text = doc_fitz[0].get_text()

            # Look for "Invoice 20561996" or "Invoice Date:"
            match = re.search(r"Invoice\s*(\d{8})", page1_text) # 8-digit invoice #
            if not match:
                # Fallback to find invoice number on its own line
                match = re.search(r"\n(\d{8})\n", page1_text)
                if match:
                    invoice_number = match.group(1)
                else:
                    # Fallback to the Delivery No if Invoice # is missing
                    match_del = re.search(r"Delivery No:\s*(\d+)", page1_text)
                    invoice_number = match_del.group(1) if match_del else "NOT_FOUND"
            else:
                 invoice_number = match.group(1)

            print(f"  > Found Invoice #: {invoice_number}")
            doc_fitz.close()

            # --- 2. Extract Text and Match with Regex ---
            extracted_items = []
            full_text = ""
            pdf_file_object = io.BytesIO(file_content)

            with pdfplumber.open(pdf_file_object) as pdf:
                # --- !! FIX !! ---
                # Loop through ALL pages, not just 1 and 2
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        full_text += page_text + "\n"

            if not full_text:
                print(f"  > ERROR: No text could be extracted from {filename}. Skipping.")
                continue

            # Go line by line, matching our regex
            for line in full_text.split('\n'):
                match = ITEM_REGEX.search(line.strip())
                if match:
                    # We found a valid item row!
                    style = match.group(1)
                    description = match.group(2)
                    quantity = match.group(3)
                    cost = match.group(4)

                    # Add the data
                    extracted_items.append([
                        invoice_number,
                        style,
                        description,
                        quantity,
                        cost
                    ])

            if not extracted_items:
                print(f"  > ERROR: No item rows matched our regex in {filename}. Skipping.")
                continue

            # --- 3. Format Data ---
            # Create the DataFrame from our extracted list
            invoice_df = pd.DataFrame(extracted_items, columns=[
                "Invoice#", "Style", "Description", "Quantity", "Cost"
            ])

            # Clean up junk (just in case)
            invoice_df = invoice_df[~invoice_df["Style"].str.contains("PO:", na=False)]
            invoice_df = invoice_df[~invoice_df["Description"].str.contains("TOTAL UNITS", na=False)]

            all_invoice_data.append(invoice_df)
            print(f"  > Successfully processed {len(invoice_df)} items.")

        except Exception as e:
            print(f"  > FATAL ERROR processing {filename}: {e}. Skipping this file.")

    print("\n--- All files processed. ---")

else:
    print("No files were uploaded to process.")

In [ ]:
# Cell 4: Combine all data, save to Excel, and download
if all_invoice_data:
    # Combine all the individual DataFrames into one big one
    master_df = pd.concat(all_invoice_data, ignore_index=True)

    output_filename = "Combined_Invoice_Data.xlsx"

    # Save the master DataFrame to an Excel file
    master_df.to_excel(output_filename, index=False)

    print(f"\nSuccessfully combined all data into '{output_filename}'.")
    print(f"Total items extracted: {len(master_df)}")
    print("\nFinal data preview:")
    print(master_df.head())

    print("\nDownloading the file to your computer...")
    files.download(output_filename)

else:
    print("\nNo data was successfully extracted, so no file will be created.")